# 05 — Insight, Rekomendasi & Laporan

Notebook ini **menghitung ulang** seluruh angka kunci dari `data/processed/` lalu menyusun:
ringkasan eksekutif, insight, rekomendasi, keterbatasan, draft LinkedIn, dan 3 poin CV.

Mencakup **dua metrik**: IKU/IKLH nasional tahunan 2020–2024 (notebook 03) dan **ISPU harian DKI Jakarta 2024–2025** (notebook 04). Semua angka diturunkan dari data; tidak ada yang dikarang.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
REP = ROOT / "reports"; REP.mkdir(exist_ok=True)

master = pd.read_csv(PROC / "provinsi_master.csv")
city = pd.read_csv(PROC / "iku_kabkota_long.csv")
prov = master[master.provinsi != "INDONESIA"].copy()

nas = prov.groupby("tahun").agg(iku=("iku", "mean"), iklh=("iklh", "mean"))
iku24 = prov[prov.tahun == 2024].copy()

piv = prov.pivot_table(index=["provinsi", "wilayah"], columns="tahun", values="iku")
piv["delta"] = piv[2024] - piv[2020]
delta = piv.dropna(subset=["delta"]).reset_index()

wil = iku24.groupby("wilayah")["iku"].mean().sort_values()
sc = iku24.dropna(subset=["kepadatan_penduduk_2021"])
r_pear, p_pear = stats.pearsonr(sc["kepadatan_penduduk_2021"], sc["iku"])

jak = city[city.provinsi == "DKI JAKARTA"].copy()
kota_adm = ["Kota Jakarta Pusat", "Kota Jakarta Utara", "Kota Jakarta Barat",
            "Kota Jakarta Selatan", "Kota Jakarta Timur"]
jak_adm = jak[jak.kabupaten_kota.isin(kota_adm)]
jpid = jak_adm.pivot_table(index="kabupaten_kota", columns="tahun", values="iku")
jpid["delta"] = jpid[2024] - jpid[2020]
jmean24 = jak_adm[jak_adm.tahun == 2024]["iku"].mean()

# --- ISPU harian Jakarta (notebook 04) ---
ispu = pd.read_csv(PROC / "jakarta_ispu_harian_clean.csv", parse_dates=["tanggal_dt"])
vj = ispu.dropna(subset=["ispu"]).copy()
vj["buruk"] = vj.kategori.isin(["TIDAK SEHAT", "SANGAT TIDAK SEHAT"])
kat = vj["kategori"].value_counts()
pol = vj["polutan_kritis"].value_counts()
st_mean = vj.groupby("kota_adm")["ispu"].mean()
st_buruk = vj.groupby("kota_adm")["buruk"].mean() * 100
mj = vj.groupby(vj.tanggal_dt.dt.to_period("M"))["ispu"].mean()
mx = vj.loc[vj.ispu.idxmax()]

isk = dict(
    n=len(vj),
    mean=vj["ispu"].mean(),
    m2024=vj[vj.tanggal_dt.dt.year == 2024]["ispu"].mean(),
    m2025=vj[vj.tanggal_dt.dt.year == 2025]["ispu"].mean(),
    n_baik=int(kat.get("BAIK", 0)), n_sedang=int(kat.get("SEDANG", 0)),
    n_tidak=int(kat.get("TIDAK SEHAT", 0)), n_sgt=int(kat.get("SANGAT TIDAK SEHAT", 0)),
    pc_baik=kat.get("BAIK", 0) / len(vj) * 100, pc_sedang=kat.get("SEDANG", 0) / len(vj) * 100,
    pc_tidak=kat.get("TIDAK SEHAT", 0) / len(vj) * 100,
    pm25=pol.get("PM2.5", 0) / pol.sum() * 100, so2=pol.get("SO2", 0) / pol.sum() * 100,
    pm10=pol.get("PM10", 0) / pol.sum() * 100,
    worst=st_mean.idxmax(), worst_v=st_mean.max(), worst_buruk=st_buruk.max(),
    best=st_mean.idxmin(), best_v=st_mean.min(),
    peak=mj.idxmax().strftime("%b %Y"), peak_v=mj.max(),
    low=mj.idxmin().strftime("%b %Y"), low_v=mj.min(),
    mx_kota=mx["kota_adm"], mx_v=mx["ispu"],
    wk=vj[~(vj.tanggal_dt.dt.dayofweek >= 5)]["ispu"].mean(),
    we=vj[vj.tanggal_dt.dt.dayofweek >= 5]["ispu"].mean(),
)

v = dict(
    iku20=nas.loc[2020, "iku"], iku24=nas.loc[2024, "iku"],
    dnas=nas.loc[2024, "iku"] - nas.loc[2020, "iku"],
    dnas_iklh=nas.loc[2024, "iklh"] - nas.loc[2020, "iklh"],
    best=iku24.loc[iku24["iku"].idxmax(), "provinsi"], best_v=iku24["iku"].max(),
    worst=iku24.loc[iku24["iku"].idxmin(), "provinsi"], worst_v=iku24["iku"].min(),
    jawa_mean=wil.get("Jawa"), papua_mean=wil.get("Papua"),
    jabar_delta=float(delta.loc[delta.provinsi == "JAWA BARAT", "delta"].iloc[0]),
    banten_delta=float(delta.loc[delta.provinsi == "BANTEN", "delta"].iloc[0]),
    r=r_pear, p=p_pear, n_dens=len(sc),
    jak_mean24=jmean24,
    jt_delta=float(jpid.loc["Kota Jakarta Timur", "delta"]),
    ju_delta=float(jpid.loc["Kota Jakarta Utara", "delta"]),
    jp_delta=float(jpid.loc["Kota Jakarta Pusat", "delta"]),
    jt_24=float(jpid.loc["Kota Jakarta Timur", 2024]),
    jp_24=float(jpid.loc["Kota Jakarta Pusat", 2024]),
)

print("== IKU nasional (2020-2024) ==")
for k in ["iku20", "iku24", "dnas", "dnas_iklh", "jawa_mean", "papua_mean", "r", "p", "n_dens"]:
    print(f"  {k:12s} = {v[k]}")
print("\n== ISPU Jakarta (2024-2025) ==")
for k, val in isk.items():
    print(f"  {k:12s} = {val}")

== IKU nasional (2020-2024) ==
  iku20        = 88.27058823529411
  iku24        = 91.32578947368421
  dnas         = 3.0552012383900973
  dnas_iklh    = 4.088312693498452
  jawa_mean    = 82.06166666666667
  papua_mean   = 95.005
  r            = -0.6661524633050072
  p            = 1.705255235050229e-05
  n_dens       = 34

== ISPU Jakarta (2024-2025) ==
  n            = 3480
  mean         = 73.36206896551724
  m2024        = 75.06304824561404
  m2025        = 71.48852657004831
  n_baik       = 444
  n_sedang     = 2694
  n_tidak      = 341
  n_sgt        = 1
  pc_baik      = 12.758620689655173
  pc_sedang    = 77.41379310344828
  pc_tidak     = 9.798850574712644
  pm25         = 86.00116414435391
  so2          = 6.5483119906868446
  pm10         = 5.122235157159487
  worst        = Jakarta Barat
  worst_v      = 76.55730659025788
  worst_buruk  = 16.332378223495702
  best         = Jakarta Timur
  best_v       = 70.81449275362318
  peak         = Jun 2025
  peak_v       = 94.82666

## Tulis laporan (ringkasan, insight, rekomendasi, LinkedIn, CV)

In [2]:
report = f'''# Analisis Kualitas Udara Indonesia
## Kualitas Udara Provinsi (IKU/IKLH 2020-2024) & Studi Kasus Harian DKI Jakarta (ISPU 2024-2025)

### Ringkasan Eksekutif
Kualitas udara Indonesia menunjukkan perbaikan yang konsisten selama 2020-2024: rata-rata IKU
provinsi naik **{v['dnas']:+.2f} poin** ({v['iku20']:.2f} -> {v['iku24']:.2f}) dan IKLH komposit +{v['dnas_iklh']:.2f} poin,
tetapi perbaikan itu tidak merata - provinsi Jawa mencatat IKU terendah ({v['jawa_mean']:.2f}) sementara Papua
tertinggi ({v['papua_mean']:.2f}), dan kualitas udara berkorelasi negatif dengan kepadatan penduduk
(Pearson r = **{v['r']:.2f}**, p < 0,001, n = {v['n_dens']}). Untuk kondisi terkini pada skala harian, studi kasus
DKI Jakarta ({isk['n']:,} pengamatan ISPU, Januari 2024 - November 2025, 5 stasiun) menunjukkan rata-rata ISPU
**{isk['mean']:.1f}** (kategori *Sedang*): {isk['pc_sedang']:.0f}% hari *Sedang* dan {isk['pc_tidak']:.1f}% hari *Tidak Sehat*, dengan
**PM2.5 sebagai pencemar kritis pada {isk['pm25']:.0f}% hari**. Semua kota administrasi Jakarta berada di bawah
rata-rata nasional versi IKU.

### Insight
1. **Tren nasional membaik.** IKU rata-rata provinsi naik dari {v['iku20']:.2f} (2020) ke {v['iku24']:.2f} (2024), {v['dnas']:+.2f} poin; IKLH +{v['dnas_iklh']:.2f} poin.
2. **Kesenjangan wilayah mencolok.** Rata-rata IKU 2024: Jawa {v['jawa_mean']:.2f} (terendah) vs Papua {v['papua_mean']:.2f} (tertinggi). Lima dari enam provinsi Jawa masuk 10 terburuk.
3. **Provinsi terbaik & terburuk 2024.** Terbaik {v['best']} ({v['best_v']:.2f}); terburuk {v['worst']} ({v['worst_v']:.2f}) - selisih {v['best_v']-v['worst_v']:.2f} poin.
4. **Korelasi kepadatan penduduk.** IKU 2024 vs kepadatan (2021): r = {v['r']:.2f} (p < 0,001). Hubungan negatif sedang-kuat: provinsi lebih padat cenderung ber-IKU lebih rendah. **Korelasi, bukan kausalitas.**
5. **Dinamika berlawanan di Jawa.** JAWA BARAT membaik {v['jabar_delta']:+.2f} poin (2020-2024), sementara BANTEN justru memburuk {v['banten_delta']:+.2f} poin - satu-satunya provinsi Jawa yang turun.
6. **Jakarta (IKU tahunan): disparitas internal besar.** Rata-rata 5 kota adm 2024 = {v['jak_mean24']:.2f}, di bawah nasional ({v['iku24']:.2f}). Jakarta Timur melonjak {v['jt_delta']:+.2f} poin (ke {v['jt_24']:.2f}), Jakarta Utara {v['ju_delta']:+.2f}, tetapi Jakarta Pusat memburuk {v['jp_delta']:+.2f} (ke {v['jp_24']:.2f}).
7. **Jakarta (ISPU harian 2024-2025): hampir selalu "Sedang".** Dari {isk['n']:,} hari-stasiun valid: **Sedang {isk['pc_sedang']:.1f}%** ({isk['n_sedang']:,}), **Baik {isk['pc_baik']:.1f}%** ({isk['n_baik']:,}), **Tidak Sehat {isk['pc_tidak']:.1f}%** ({isk['n_tidak']:,}), **Sangat Tidak Sehat {isk['n_sgt']} hari**. Rata-rata ISPU {isk['mean']:.1f}; 2024 ({isk['m2024']:.1f}) cenderung lebih buruk daripada 2025 Jan-Nov ({isk['m2025']:.1f}).
8. **Pencemar kritis didominasi PM2.5.** PM2.5 menjadi parameter kritis pada **{isk['pm25']:.1f}%** hari, disusul SO2 ({isk['so2']:.1f}%) dan PM10 ({isk['pm10']:.1f}%). Artinya masalah Jakarta bukan SO2/NO2 (basis IKU), melainkan partikulat halus.
9. **Barat terburuk, Timur terbaik.** Rata-rata ISPU & % hari Tidak Sehat: {isk['worst']} **{isk['worst_v']:.1f}** ({isk['worst_buruk']:.1f}%) tertinggi; {isk['best']} **{isk['best_v']:.1f}** terendah. Hari ekstrem: {isk['mx_kota']} ISPU {isk['mx_v']:.0f} (Sangat Tidak Sehat).
10. **Pola musiman kuat, akhir pekan tidak berbeda.** ISPU memuncak pada musim kemarau ({isk['peak']} = {isk['peak_v']:.1f}) dan terendah pada musim hujan ({isk['low']} = {isk['low_v']:.1f}). Rata-rata hari kerja ({isk['wk']:.2f}) hampir identik dengan akhir pekan ({isk['we']:.2f}) - lalu lintas bukan satu-satunya penggerak.

### Rekomendasi
1. **Prioritaskan PM2.5.** Karena {isk['pm25']:.0f}% hari kritis disebabkan PM2.5, intervensi harus menyasar sumber partikulat (transportasi diesel, pembakaran, industri), bukan hanya NO2/SO2.
2. **Fokuskan wilayah {isk['worst']}** yang mencatat ISPU rata-rata dan frekuensi hari Tidak Sehat tertinggi.
3. **Lanjutkan pemantauan harian:** ISPU harian mengungkap episode yang tak terlihat pada IKU tahunan; pertahankan minimal 5 SPKU dan publikasi terbuka.
4. **Antisipasi musim kemarau (Mei-Oktober):** siapkan peringatan dini dan pembatasan sumber emisi pada bulan puncak.
5. **Perkuat wilayah padat nasional:** perluas transportasi publik, uji emisi, dan pengawasan kawasan industri di BANTEN, DKI JAKARTA, dan JAWA BARAT.
6. **Perluas metrik resmi:** jika memungkinkan, masukkan PM2.5/PM10 ke dalam IKU nasional agar selaras dengan risiko kesehatan.

### Keterbatasan
- **Dua metrik berbeda, tidak dicampur.** IKU (skala 0-100, **makin tinggi makin baik**, dari NO2 & SO2, **tahunan**) != ISPU (skala 0-500+, **makin tinggi makin buruk**, **harian**).
- **Periode ISPU berakhir 30 November 2025**; belum ada data 2026 pada sumber resmi. Nilai 2025 bersifat Januari-November.
- **Cakupan ISPU terbatas 5 stasiun** (DKI1-DKI5) mewakili 5 kota administrasi; Kab. Adm. Kepulauan Seribu tanpa stasiun tetap.
- **Kualitas data:** 13 baris bernilai ISPU 0 tanpa polutan apa pun adalah placeholder "tidak ada data" (dikosongkan); 20 dari 3.500 baris ISPU kosong; 1 stasiun-hari duplikat dibuang.
- **Kepadatan penduduk hanya 2021** dan 34 provinsi lama.
- **Kendaraan bermotor tidak dianalisis** (data tidak dapat diotomatisasi dari sumber publik).
- 15 dari 514 kab/kota gagal diekstrak dari PDF sumber.
- Analisis korelasi **tidak** membuktikan hubungan sebab-akibat.

### Draft LinkedIn (<=150 kata)
Kualitas udara Indonesia membaik, tapi belum merata. Saya menganalisis data resmi KLHK, BPS, dan
Satu Data Jakarta dengan Python (pandas, matplotlib, scipy).

IKU provinsi naik {v['dnas']:+.2f} poin (2020-2024), namun Jawa tetap terendah ({v['jawa_mean']:.1f}) vs Papua ({v['papua_mean']:.1f});
IKU berkorelasi negatif dengan kepadatan penduduk (r = {v['r']:.2f}, p < 0,001). Di DKI Jakarta, saya mengolah
{isk['n']:,} pengamatan ISPU harian (2024-Nov 2025, 5 stasiun): rata-rata {isk['mean']:.1f} (kategori Sedang) - {isk['pc_sedang']:.0f}% hari
Sedang, {isk['pc_tidak']:.1f}% Tidak Sehat, dan PM2.5 menjadi pencemar kritis pada {isk['pm25']:.0f}% hari. {isk['worst']} terburuk;
polusi memuncak saat kemarau ({isk['peak']}) dan tak berbeda antara hari kerja dan akhir pekan.

Perbaikan nasional nyata, tetapi kota padat tetap jadi tantangan utama. Korelasi bukan kausalitas.

#DataAnalytics #Python #KualitasUdara #Lingkungan

### Poin CV
- Menganalisis kualitas udara **38 provinsi (IKU/IKLH 2020-2024)** dan **{isk['n']:,} pengamatan ISPU harian DKI Jakarta (2024-2025)** dari sumber resmi KLHK, BPS, dan Satu Data Jakarta; melakukan ekstraksi PDF serta parsing API/tabel JavaScript dengan Python (pdfplumber, requests, pandas).
- Membangun pipeline data 3 tahap (collection, cleaning, EDA) dengan verifikasi silang dua sumber (selisih IKU <= 0,03 poin) dan dokumentasi sumber data lengkap.
- Menemukan korelasi negatif kepadatan penduduk-IKU 2024 (r = {v['r']:.2f}, p < 0,001) serta mengidentifikasi **PM2.5 sebagai pencemar kritis pada {isk['pm25']:.0f}% hari** di Jakarta; menyajikan **12 visualisasi** dan rekomendasi berbasis data.
'''

out_md = REP / "RINGKASAN_LENGKAP.md"
out_md.write_text(report, encoding="utf-8")
print("Laporan tersimpan:", out_md)
print("=" * 70)
print(report)

Laporan tersimpan: C:\Voicegame\analisis-kualitas-udara-indonesia\reports\RINGKASAN_LENGKAP.md
# Analisis Kualitas Udara Indonesia
## Kualitas Udara Provinsi (IKU/IKLH 2020-2024) & Studi Kasus Harian DKI Jakarta (ISPU 2024-2025)

### Ringkasan Eksekutif
Kualitas udara Indonesia menunjukkan perbaikan yang konsisten selama 2020-2024: rata-rata IKU
provinsi naik **+3.06 poin** (88.27 -> 91.33) dan IKLH komposit +4.09 poin,
tetapi perbaikan itu tidak merata - provinsi Jawa mencatat IKU terendah (82.06) sementara Papua
tertinggi (95.00), dan kualitas udara berkorelasi negatif dengan kepadatan penduduk
(Pearson r = **-0.67**, p < 0,001, n = 34). Untuk kondisi terkini pada skala harian, studi kasus
DKI Jakarta (3,480 pengamatan ISPU, Januari 2024 - November 2025, 5 stasiun) menunjukkan rata-rata ISPU
**73.4** (kategori *Sedang*): 77% hari *Sedang* dan 9.8% hari *Tidak Sehat*, dengan
**PM2.5 sebagai pencemar kritis pada 86% hari**. Semua kota administrasi Jakarta berada di bawah
rata-rata nasi

## Deliverable
- `reports/RINGKASAN_LENGKAP.md` — ringkasan eksekutif, insight, rekomendasi, keterbatasan, draft LinkedIn, poin CV.
- 12 gambar di `images/` (6 `03_*` IKU nasional + 6 `04_ispu_*` Jakarta).
- Data analysis-ready di `data/processed/`.